In [0]:
from pyspark.sql.functions import sum, col, avg, coalesce, current_timestamp
from pyspark.sql.window import Window

# =========================
# 1. LECTURE BRONZE
# =========================
bronze_df = spark.read.table("iotmlhealthcatalog.bronze.vitaldbtrain")

# =========================
# 2. RENOMMAGE PROPRE
# =========================
renamed_df = bronze_df.select(
    col("caseid"),
    col("timestamp"),
    col("SNUADC_ART_SBP").alias("sbp"),
    col("Solar8000_HR").alias("hr"),
    col("Solar8000_PLETH_SPO2").alias("spo2"),
    col("Solar8000_BT").alias("temp"),
    col("target"),
    col("Device_Battery_Level"),
    col("Operator_ID")
)

# =========================
# 3. SUPPRESSION COLONNES NON UTILES
# =========================
clean_df = renamed_df.drop("Device_Battery_Level", "Operator_ID")

# =========================
# 4. WINDOW PAR PATIENT
# =========================
clean_df = clean_df.repartition("caseid")
window_case = Window.partitionBy("caseid")

cols_to_fix = ["sbp", "hr", "spo2", "temp"]

# =========================
# 5. IMPUTATION ROBUSTE
# =========================
silver_df = clean_df

for c in cols_to_fix:
    
    # moyenne par patient
    case_avg = avg(col(c)).over(window_case)

    # fallback global si patient totalement NULL
    global_avg = avg(col(c)).over(Window.partitionBy())

    silver_df = silver_df.withColumn(
        c,
        coalesce(col(c), case_avg, global_avg)
    )

# =========================
# 6. FEATURE FINAL
# =========================
silver_df = silver_df.withColumn("ingestion_timestamp", current_timestamp())

# =========================
# 7. VALIDATION (IMPORTANT)
# =========================
print("Null count check Before Imputation:")
clean_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in cols_to_fix
]).show()

print("Null count check After Imputation:")
silver_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in cols_to_fix
]).show()

# =========================
# 8. WRITE SILVER
# =========================
silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("iotmlhealthcatalog.silver.vitaldbtrain")

print("✅ Silver table created successfully")